# 05 — Label Generation

**PrioMed-Rec | Sub-Task 5**

Generates synthetic medication reconciliation cases as the model target `y`.

**SYNTHETIC DATA DISCLAIMER:** Labels are engineered proxies — not clinical gold-standard errors.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.data.admission_cohort import build_admission_cohort
from src.data.medication_timeline import (
    build_medication_timeline, build_pre_admission_meds, build_admission_meds
)
from src.labels.synthetic_case_generator import generate_cases, save_synthetic_cases, RANDOM_SEED_DEFAULT
print('Seed:', RANDOM_SEED_DEFAULT)

## 1. Load cohort and medication data

In [ ]:
cohort   = build_admission_cohort('../data/raw/synthea')
timeline = build_medication_timeline('../data/raw/synthea')
pre_meds = build_pre_admission_meds(cohort, timeline)
adm_meds = build_admission_meds(cohort, timeline)
print('Cohort:', len(cohort), '| Pre:', len(pre_meds), '| Adm:', len(adm_meds))

## 2. Generate cases

In [ ]:
cases = generate_cases(cohort, pre_meds, adm_meds, seed=RANDOM_SEED_DEFAULT)
print('Total cases:', len(cases))
cases.head()

## 3. Label and mutation distribution

In [ ]:
print('Label distribution (0=NONE, 1=discrepancy):')
print(cases['label'].value_counts().to_string())
print()
print('Mutation type distribution:')
print(cases['mutation_type'].value_counts().to_string())
print()
print('Class balance (%):')
print((cases['label'].value_counts(normalize=True)*100).round(1).to_string())

## 4. Reproducibility check

In [ ]:
cases2 = generate_cases(cohort, pre_meds, adm_meds, seed=RANDOM_SEED_DEFAULT)
assert cases.equals(cases2), 'REPRODUCIBILITY FAILED'
print('Reproducibility check: PASSED')

## 5. Build model_ready.csv

In [ ]:
fm = pd.read_csv('../data/processed/feature_matrix.csv', index_col='encounter_id')
label_per_enc = cases.groupby('encounter_id')['label'].max().rename('has_discrepancy')
model_ready = fm.join(label_per_enc, how='left')
model_ready['has_discrepancy'] = model_ready['has_discrepancy'].fillna(0).astype(int)
def assign_priority(row):
    if row['high_alert_flag'] == 1 or row['new_medication_count'] >= 3:
        return 2
    if (row['polypharmacy_flag'] == 1 or row['discontinued_medication_count'] >= 2
            or row['active_condition_count'] >= 3):
        return 1
    return 0
model_ready['review_priority'] = model_ready.apply(assign_priority, axis=1)
model_ready.to_csv('../data/processed/model_ready.csv')
print('model_ready saved:', model_ready.shape)
print(model_ready[['has_discrepancy','review_priority']].describe().to_string())

## 6. Save synthetic cases

In [ ]:
save_synthetic_cases(cases, '../data/processed/synthetic_cases.csv')
print('Done.')